In [11]:
from mace.calculators import MACECalculator
import numpy as np
import pandas as pd
import ase
from ase import Atom, Atoms

from ase.build import graphene
from ase.optimize import BFGS

from ase.filters import ExpCellFilter

from ase.visualize import view

from ase.io import write

import matplotlib.pyplot as plt

from tqdm import tqdm
import spglib

from ase.spacegroup.symmetrize import check_symmetry

In [12]:
model = MACECalculator("../../../../MACE.model", device = "cuda")

Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [13]:
primitive = graphene()
primitive.cell[2] = [0,0,100]
filter = ExpCellFilter(primitive, mask = [1,1,0,0,0,0])
primitive.calc = model

opt = BFGS(filter)
opt.run(fmax = 1e-5)

/tmp/ipykernel_7787/1072062919.py:3: DeprecationWarning: Use FrechetCellFilter for better convergence w.r.t. cell variables.
  filter = ExpCellFilter(primitive, mask = [1,1,0,0,0,0])


      Step     Time          Energy          fmax
BFGS:    0 09:32:04      -15.917213        0.014590
BFGS:    1 09:32:04      -15.917210        0.029133
BFGS:    2 09:32:04      -15.917217        0.000016
BFGS:    3 09:32:04      -15.917215        0.000018
BFGS:    4 09:32:04      -15.917215        0.000035
BFGS:    5 09:32:04      -15.917215        0.000029
BFGS:    6 09:32:04      -15.917215        0.000004


np.True_

In [14]:
cell0 = primitive.cell.copy()
a0 = np.linalg.norm(cell0[0])
l0 = primitive.get_all_distances()[0][1]

In [15]:
KCELL0 = 3*l0* np.array([
    [1,0,0],
    [0.5, np.sqrt(3)/2, 0],
    [0,0,100]
])
# in K cell 0 primitive direction
KBASE0 = (1/3.) * np.array([
    [1, 0, 0],
    [2, 0, 0],
    [2, 1, 0],
    [1, 2, 0],
    [0, 1, 0],
    [0, 2, 0]
])

katoms = Atoms("C6", positions = KBASE0 @ KCELL0, cell=KCELL0, pbc = [True, True, True])
katoms.cell[2] = [0,0,100]


In [20]:
katoms.cell

Cell([[4.260548719479057, 0.0, 0.0], [2.1302743597395284, 3.689743425130123, 0.0], [0.0, 0.0, 100.0]])

In [18]:
check_symmetry(katoms)

SpglibDataset(number=191, hall_number=485, international='P6/mmm', hall='-P 6 2', choice='', transformation_matrix=array([[2., 1., 0.],
       [1., 2., 0.],
       [0., 0., 1.]]), origin_shift=array([0., 0., 0.]), rotations=array([[[ 1,  0,  0],
        [ 0,  1,  0],
        [ 0,  0,  1]],

       [[-1,  0,  0],
        [ 0, -1,  0],
        [ 0,  0, -1]],

       [[ 0, -1,  0],
        [ 1,  1,  0],
        [ 0,  0,  1]],

       [[ 0,  1,  0],
        [-1, -1,  0],
        [ 0,  0, -1]],

       [[-1, -1,  0],
        [ 1,  0,  0],
        [ 0,  0,  1]],

       [[ 1,  1,  0],
        [-1,  0,  0],
        [ 0,  0, -1]],

       [[-1,  0,  0],
        [ 0, -1,  0],
        [ 0,  0,  1]],

       [[ 1,  0,  0],
        [ 0,  1,  0],
        [ 0,  0, -1]],

       [[ 0,  1,  0],
        [-1, -1,  0],
        [ 0,  0,  1]],

       [[ 0, -1,  0],
        [ 1,  1,  0],
        [ 0,  0, -1]],

       [[ 1,  1,  0],
        [-1,  0,  0],
        [ 0,  0,  1]],

       [[-1, -1,  0],
      

In [19]:
supercell = primitive.repeat((3,3,1))
supercell.cell[2] = [0,0,100]

In [21]:
check_symmetry(supercell)

SpglibDataset(number=191, hall_number=485, international='P6/mmm', hall='-P 6 2', choice='', transformation_matrix=array([[3., 0., 0.],
       [0., 3., 0.],
       [0., 0., 1.]]), origin_shift=array([0.66666665, 0.33333334, 0.        ]), rotations=array([[[ 1,  0,  0],
        [ 0,  1,  0],
        [ 0,  0,  1]],

       [[-1,  0,  0],
        [ 0, -1,  0],
        [ 0,  0, -1]],

       [[ 1, -1,  0],
        [ 1,  0,  0],
        [ 0,  0,  1]],

       ...,

       [[-1,  0,  0],
        [-1,  1,  0],
        [ 0,  0,  1]],

       [[ 1, -1,  0],
        [ 0, -1,  0],
        [ 0,  0, -1]],

       [[-1,  1,  0],
        [ 0,  1,  0],
        [ 0,  0,  1]]], shape=(216, 3, 3), dtype=int32), translations=array([[-0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
       [ 5.55555570e-01,  7.77777772e-01,  0.00000000e+00],
       [ 8.88888886e-01,  1.11111101e-01,  0.00000000e+00],
       [ 6.66666684e-01,  6.66666671e-01,  0.00000000e+00],
       [ 6.66666671e-01,  9.99999987e-01,  0.

In [22]:
from ase.utils.structure_comparator import SymmetryEquivalenceCheck

In [24]:
comp = SymmetryEquivalenceCheck()

In [25]:
comp.compare(katoms, supercell)

False

In [26]:
comp.compare(supercell, primitive)

False

In [ ]:
from spglib import spglib

In [40]:
spglib.get_spacegroup(
    (
        np.asarray(primitive.cell / np.linalg.norm(primitive.cell, axis=0)),
        np.asarray(primitive.get_scaled_positions()),
        np.asarray(primitive.numbers),
    ),
    symprec = 1e-5
)

'Cmmm (65)'

In [41]:
katoms

Atoms(symbols='C6', pbc=True, cell=[[4.260548719479057, 0.0, 0.0], [2.1302743597395284, 3.689743425130123, 0.0], [0.0, 0.0, 100.0]])

In [42]:
primitive

Atoms(symbols='C2', pbc=[True, True, False], cell=[[2.459828913293515, 0.0, 0.0], [-1.2299144566467575, 2.1302741845580493, 0.0], [0.0, 0.0, 100.0]], calculator=MACECalculator(...))